# 03 – Privacy & Governance Assessment

**Role:** Governance Officer  
**Task Force:** NovaCred  

This notebook serves as the primary governance audit for the NovaCred credit scoring pipeline. It evaluates the dataset for privacy risks, identifies Personally Identifiable Information (PII), demonstrates data protection techniques, and maps technical findings to the **GDPR** and the **EU AI Act**.

## 1. Comprehensive PII & Sensitivity Audit (Part A)

Based on the structural inspection of the `raw_credit_applications.json` dataset and the downstream analyses performed by the Data Engineering and Data Science teams, we have conducted a full forensic audit of the data collected by NovaCred.

### 1.1 Direct Identifiers
These fields can uniquely and directly identify a data subject. Under GDPR, they require the highest level of security (Art. 32).
- **`full_name` & `email`:** Primary identifiers used for communication. They present a standard privacy risk but are necessary for contract fulfillment (Art. 6(1)(b)).
- **`ssn` (Social Security Number):** A highly sensitive national identifier. Storing this in plain text within the raw zone is a severe security vulnerability that exposes applicants to identity theft.

### 1.2 Quasi-identifiers & Re-identification Risk
While these fields do not directly identify an individual on their own, their combination creates a unique fingerprint (Re-identification Risk).
- **`ip_address`:** Collected during the application process. Under GDPR, dynamic IP addresses are considered personal data. This field violates the **Data Minimization principle (Art. 5(1)(c))** as it is not strictly necessary for evaluating creditworthiness.
- **`date_of_birth` & `zip_code`:** These are classic quasi-identifiers. More critically, as demonstrated in `02-bias-analysis.ipynb`, the Data Scientist team found that the `zip_code` acts as a **geographic proxy for protected attributes**, directly contributing to the Disparate Impact Ratio (DIR) failure. Retaining this field actively harms the algorithmic fairness of the system.

### 1.3 Special Categories of Data (Art. 9 GDPR Violations)
An inspection of the nested `spending_behavior` array revealed alarming data collection practices.
- The algorithm ingests raw spending categories, including sensitive tags such as **"Alcohol"** (observed in application `app_200`).
- **Governance Implication:** Tracking alcohol consumption or similar habits infers health data and potential addictions. Under **Article 9 of the GDPR**, processing data concerning health is strictly prohibited without explicit consent. Using this to calculate an `algorithm_risk_score` is a severe compliance breach.

### 1.4 Accuracy and Storage Integrity
According to the `01-data-quality.ipynb` audit, the raw dataset contained 491 records, of which only 485 formed a clean baseline (due to duplicates and invalid entries).
- **Governance Implication:** Processing duplicate or corrupted financial records violates the **Principle of Accuracy (Art. 5(1)(d) GDPR)**, leading to unjustified automated credit rejections.

## 2. Pseudonymization vs Anonymization (Part B)

To comply with **Art. 25 GDPR (Privacy by Design)**, we must protect sensitive fields like the SSN. It is crucial to distinguish between two concepts:
1. **Anonymization:** Irreversibly destroying the link to the individual (e.g., dropping the SSN column entirely or aggregating data). Anonymized data is no longer subject to GDPR.
2. **Pseudonymization:** Replacing direct identifiers with artificial identifiers (e.g., hashing). The data can still be re-identified using an external "key". This data remains subject to GDPR but significantly reduces security risks.

Below, we demonstrate the **Pseudonymization** of the SSN using a SHA-256 hashing algorithm, which should be implemented at the Data Lake ingestion layer (Raw Zone).

In [ ]:
import hashlib
import pandas as pd

# 1. Create a mock dataframe representing the raw data
df_mock = pd.DataFrame({
    'applicant_id': ['app_200', 'app_306'],
    'full_name': ['Jerry Smith', 'Anna White'],
    'ssn': ['596-64-4340', '757-27-8131']
})

# 2. Define the Pseudonymization function
def hash_ssn(ssn_string):
    """Applies SHA-256 hashing to a string to pseudonymize it."""
    if pd.isna(ssn_string): return None
    # In production, a cryptographic 'salt' should be added to prevent dictionary attacks
    return hashlib.sha256(str(ssn_string).encode('utf-8')).hexdigest()

# 3. Apply the transformation
df_mock['ssn_pseudonymized'] = df_mock['ssn'].apply(hash_ssn)

# 4. Display the results
print("--- Data Protection Demonstration ---")
display(df_mock[['applicant_id', 'ssn', 'ssn_pseudonymized']])
print("\nNotice: The original SSN must be dropped from the analytics layer after this transformation.")

## 3. GDPR & EU AI Act Remediation Plan (Part C)

Based on the combined technical audits, NovaCred is currently exposed to significant regulatory risk. Below is the Governance Task Force's remediation plan.

### 3.1 EU AI Act Requirements
**Classification:** The NovaCred machine learning model is classified as a **High-Risk AI System** under Annex III (Credit scoring of natural persons).

**Mandatory Interventions:**
- **Human Oversight (Art. 14):** The current system uses an `algorithm_risk_score` for automated rejections. We must establish a "Credit Review Committee" (Human-in-the-loop) to review automated denials and allow applicants to contest the decision.
- **Transparency (Art. 13):** Applicants must be explicitly informed that an AI system is processing their data and assessing their creditworthiness.

### 3.2 GDPR Actionable Controls
1. **Drop Invalid Variables (Art. 5 Data Minimization):** The `ip_address` must be removed from the ingestion pipeline as it provides no legitimate value for credit scoring.
2. **Mitigate Proxy Discrimination:** The `zip_code` must be removed from the model training features to resolve the Disparate Impact Ratio failure found in `02-bias-analysis`.
3. **Consent for Behavioral Data (Art. 9):** Processing categories like "Alcohol" must be completely purged from the dataset. If tracking spending behavior is strictly necessary, it must be aggregated into high-level, non-sensitive categories (e.g., "Groceries", "Entertainment").
4. **Data Retention Policy:** Implement an automated job to delete the records of rejected applicants after a legally justified period (e.g., 5 years for audit purposes), fulfilling the storage limitation principle.